<a href="https://colab.research.google.com/github/shahad030/Final-Project--Bayan-bilingual-citizen-feedback/blob/main/Copy_of_Student_Lab3a_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3a — Train the Bayan Classifier | تدريب مصنّف بيان
**SDA-AIE-211 | Day 2 **

## القصة | The story
- 🇸🇦 تصل منصة بيان رسائل كثيرة، ونريد توجيه كل رسالة إلى **الموضوع الصحيح** تلقائيًا (فواتير، مياه، نقل...). لكن قبل أن نثق بنموذج قوي، يجب أن نثبت أنه **أفضل فعلًا** من طريقة بسيطة (TF-IDF).
- 🇬🇧 Bayan receives many messages and we want to route each to the **right topic** (billing, water, transport…). Before trusting a strong model, we must prove it is **really better** than a simple one (TF-IDF).

## الخطوات | Steps
| # | الخطوة | Step |
|---|---|---|
| 1 | تحميل البيانات | Load data |
| 2 | التقسيم حسب المواطن (لمنع التسريب) | Split by citizen (no leakage) |
| 3 | النموذج الأساسي TF-IDF | TF-IDF baseline |
| 4 | تدريب Transformer | Train the Transformer |
| 5 | الاختبار النهائي مرة واحدة + حفظ الملفات | One final test + save files |

**Macro-F1:** متوسط الأداء على كل المواضيع بالتساوي / average score over all topics equally.
⏱️ 50 min · 🖥️ فعّل GPU: *Runtime → Change runtime type → GPU*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path

# Choose checkpoint(s) using measured fertility, p95 length, and language coverage
decision = {
    'arabic_checkpoint': 'CAMeL-Lab/bert-base-arabic-camelbert-mix',
    'bilingual_checkpoint': 'xlm-roberta-base',
    'evidence': (
        'CAMeLBERT has the best Arabic efficiency (fertility 1.44, p95 35) but is poor on English '
        '(fertility 2.51, p95 73). XLM-R is the most balanced across both languages '
        '(fertility ar 1.71 / en 1.37, p95 41 / 41) and beats mBERT on Arabic '
        '(mBERT: 2.29, p95 56). Use XLM-R as the single bilingual model with max_length 64, '
        'which covers p95 with headroom.'
    )
}

# Print the decision as formatted JSON
print(json.dumps(decision, indent=2, ensure_ascii=False))

# حفظ الملفات للابات القادمة | Save files for the next labs
# Local writable folder (the Drive folder is read-only)
ARTIFACT_DIR = Path('/content/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(ARTIFACT_DIR / 'lab1_decision.json', 'w', encoding='utf-8') as f:
    json.dump(decision, f, ensure_ascii=False, indent=2)


print('✅ Saved decision + clean data')

{
  "arabic_checkpoint": "CAMeL-Lab/bert-base-arabic-camelbert-mix",
  "bilingual_checkpoint": "xlm-roberta-base",
  "evidence": "CAMeLBERT has the best Arabic efficiency (fertility 1.44, p95 35) but is poor on English (fertility 2.51, p95 73). XLM-R is the most balanced across both languages (fertility ar 1.71 / en 1.37, p95 41 / 41) and beats mBERT on Arabic (mBERT: 2.29, p95 56). Use XLM-R as the single bilingual model with max_length 64, which covers p95 with headroom."
}
✅ Saved decision + clean data


Before starting this lab, examine the complaint dataset and make a prediction: Which two complaint topics do you think the model is most likely to confuse, and why?

قبل البدء في هذا المختبر، تفحّص بيانات الشكاوى وحاول التنبؤ: ما نوعا الشكاوى اللذان تعتقد أن النموذج قد يخلط بينهما أكثر من غيرهما؟ ولماذا؟

## Step 1 — تحميل البيانات | Load data

In [ ]:
!pip -q install pandas numpy scikit-learn torch transformers datasets accelerate

from pathlib import Path
import zipfile, json, re
import numpy as np
import pandas as pd

DATA_DIR = Path('/content/drive/MyDrive/NATURAL LANGUAGE PROCESSING WITH TRANSFORMER/Data')           # مجلد البيانات | data folder
ARTIFACT_DIR = Path('/content/artifacts')  # مجلد النتائج | results folder
ARTIFACT_DIR.mkdir(exist_ok=True)


feedback = pd.read_csv(DATA_DIR / '/content/bayan_feedback_clean.csv')


import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

feedback[['text_clean', 'topic']].head(3)

,text_clean,topic
0,"Honestly, The card payment went through yet I ...",billing
1,Update: Stray cats are tearing open the rubbis...,waste
2,للعلم، اضطررنا لطلب وايت لأن الصنابير جافة. أر...,outage


## Step 2 — التقسيم حسب المواطن | Split by citizen
- 🇸🇦 المواطن الواحد قد يرسل رسائل متشابهة. لو ظهرت رسائله في التدريب **والاختبار** فسيحفظ النموذج الإجابة (**تسريب**). الحل: **كل رسائل المواطن في قسم واحد فقط**.
- 🇬🇧 One citizen may send near-identical messages. If they appear in train **and** test the model just memorizes (**leakage**). Fix: **each citizen lives in one split only**.
- الأقسام / Splits: **Train** تدريب · **Valid** تحقق (لاختيار النموذج) · **Frozen test** اختبار مجمّد (نلمسه مرة واحدة).

In [ ]:
# ✏️ TODO 1: المجموعات = عمود المواطن 'citizen_id' | groups = the 'citizen_id' column
groups_all = feedback['citizen_id']

# 1) افصل 20% اختبارًا مجمّدًا | hold out 20% as frozen test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(feedback, feedback['topic'], groups=groups_all))
train_pool, frozen_test = feedback.iloc[train_idx].copy(), feedback.iloc[test_idx].copy()

# 2) من الباقي افصل 15% للتحقق | from the rest take 15% for validation
splitter_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=7)
tr_idx, va_idx = next(splitter_val.split(train_pool, train_pool['topic'], groups=train_pool['citizen_id']))
train, valid = train_pool.iloc[tr_idx].copy(), train_pool.iloc[va_idx].copy()

print('Rows (train, valid, test):', len(train), len(valid), len(frozen_test))
# ✏️ TODO 2: احسب عدد المواطنين المشتركين بين train و valid (يجب أن يكون 0)
# Count citizens shared by train and valid (must be 0)
overlap = len(set(train['citizen_id']) & set(valid['citizen_id']))
print('Citizen overlap / المشتركون:', overlap)
assert overlap == 0, 'يوجد تسريب! | Leakage found!'

Rows (train, valid, test): 8244 1362 2394
Citizen overlap / المشتركون: 0


## Step 3 — النموذج الأساسي | TF-IDF baseline
- 🇸🇦 TF-IDF يحوّل الكلمات إلى أرقام، و`LinearSVC` يتعلم التصنيف. بسيط وسريع، وهو **مسطرة المقارنة**.
- 🇬🇧 TF-IDF turns words into numbers and `LinearSVC` learns the classes. Simple and fast — our **yardstick**.

In [ ]:
vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2) #	الاحتفاظ بحد أقصى 30,000 خاصية لغوية.، 	استخدام الكلمات المفردة والأزواج المتجاورة من الكلمات.، 	تجاهل الكلمات أو العبارات التي تظهر في مستند واحد فقط.، n_gram = هذا الإعداد يحدد عدد الكلمات التي نأخذها معًا عند تكوين Features.وحدة و اثنين, ا تحتفظ بالكلمة أو الـ N-gram إلا إذا ظهرت في مستندين على الأقل.
X_train = vectorizer.fit_transform(train.text_clean)     # fit على التدريب فقط | fit on train ONLY
X_valid = vectorizer.transform(valid.text_clean)

baseline = LinearSVC().fit(X_train, train.topic)
baseline_f1 = f1_score(valid.topic, baseline.predict(X_valid), average='macro')
print(f'TF-IDF validation macro-F1: {baseline_f1:.3f}')   # ✍️ سجّل الرقم | write it down
if baseline_f1 > 0.95:
    print('⚠️ الأساس شبه مثالي: بيانات الدورة اصطناعية ومتكررة القوالب، لذا يصعب على أي نموذج التفوق عليه.')
    print('⚠️ Baseline is near-perfect: the course data is synthetic and template-based, so no model can beat it by +0.08.')

TF-IDF validation macro-F1: 0.820


## Step 4 — تدريب Transformer | Train the Transformer
- 🇸🇦 نستخدم نموذجًا **مدرَّبًا مسبقًا** ونكمل تدريبه على بياناتنا (Transfer Learning). بعد كل epoch نقيس الأداء ونحتفظ بالأفضل.
- 🇬🇧 We reuse a **pretrained** model and fine-tune it on our data (Transfer Learning). After each epoch we score it and keep the best.
- 🔒 كود التدريب **جاهز للقراءة فقط**؛ المطلوب فقط اختيار `CHECKPOINT`. / Training code is **read-only**; only choose `CHECKPOINT`.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
# ✏️ TODO 3: النموذج ثنائي اللغة الذي اخترته في Lab 1 | the bilingual checkpoint from Lab 1
CHECKPOINT = 'xlm-roberta-base'
MAX_LENGTH = 128 # Maximum number of tokens processed per text
EPOCHS = 1

labels = sorted(feedback.topic.dropna().unique().tolist()) # استخراج أسماء التصنيفات الموجودة في البيانات دون تكرار
label_to_id = {name: i for i, name in enumerate(labels)} #مثال: {'billing': 0, 'transport': 1, 'water': 2}
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT) #Load the tokenizer associated with XLM-RoBERTa
# Define a function to convert a DataFrame into a Hugging Face Dataset
def to_dataset(frame):
    ds = Dataset.from_dict({'text': frame.text_clean.tolist(),
                            'labels': [label_to_id[x] for x in frame.topic]})
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LENGTH), batched=True)

train_ds, valid_ds = to_dataset(train), to_dataset(valid)
model = AutoModelForSequenceClassification.from_pretrained( # Load the pre-trained XLM-RoBERTa model for sequence classification
    CHECKPOINT, num_labels=len(labels),
    id2label={i: x for i, x in enumerate(labels)}, label2id=label_to_id)  # Map numerical IDs back to topic names

def compute_metrics(eval_pred): #Extract the model's prediction scores (logits) and the true labels
    logits, y = eval_pred  #Select the class with the highest score for each text
    pred = np.argmax(logits, axis=1)
    return {'macro_f1': f1_score(y, pred, average='macro'), 'accuracy': accuracy_score(y, pred)}

# Configure the training process and evaluation settings
args = TrainingArguments(
    output_dir='/content/bayan_topic_model', learning_rate=2e-5, # Directory to save model checkpoints
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1',
    fp16=torch.cuda.is_available(), report_to='none', save_total_limit=1) # Use mixed-precision training if a CUDA GPU is available


# Create the Trainer to manage model training and evaluation
trainer = Trainer(
    model=model,  # Pretrained model with a classification head
    args=args,  # Training configuration
    train_dataset=train_ds,  # Dataset used to train the model
    eval_dataset=valid_ds,  # Dataset used to evaluate the model during training
    data_collator=DataCollatorWithPadding(tokenizer),  # يضيف [PAD] للجمل القصيرة لتتساوى أطوال الدفعة | Pads short texts so every batch has equal length
    compute_metrics=compute_metrics  # Function used to calculate evaluation metrics
)


# Start fine-tuning the pretrained model on the Bayan topic classification dataset
trainer.train()


# Save the fine-tuned model and its configuration
trainer.save_model('/content/bayan_topic_model')

# Save the tokenizer files needed to preprocess new text in the same way
tokenizer.save_pretrained('/content/bayan_topic_model')


# Evaluate the best model on the validation dataset and display the final metrics
print(trainer.evaluate())

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/8244 [00:00<?, ? examples/s]

Map:   0%|          | 0/1362 [00:00<?, ? examples/s]

... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1,Accuracy
0.907164,0.422296,1,0.879539,0.876652


{'eval_loss': 0.42229604721069336, 'eval_macro_f1': 0.8795387852680852, 'eval_accuracy': 0.8766519823788547}


## Step 5 — الاختبار النهائي + الحفظ | One final test + save
⚠️ **القاعدة الذهبية / Golden rule:**
- 🇸🇦 الاختبار المجمّد يُستخدم **مرة واحدة فقط**، ولا نعدّل النموذج بناءً على نتيجته.
- 🇬🇧 The frozen test is used **only once**; never tune the model using its score.

In [ ]:
# ==========================================
# الاختبار النهائي | Frozen Test
# نستخدم بيانات الاختبار مرة واحدة فقط
# ==========================================

# استخدام النموذج للتنبؤ بموضوع كل رسالة في بيانات الاختبار
# Predict the topic of each test message
test_output = trainer.predict(to_dataset(frozen_test))

# استخراج التصنيف صاحب أعلى درجة لكل رسالة
# Select the class with the highest prediction score
test_ids = np.argmax(test_output.predictions, axis=1)


# اختيار الأعمدة الموجودة فعلًا في ملف الاختبار
# Keep only the columns that exist in the test dataset
keep = [
    c for c in [
        'feedback_id', 'citizen_id', 'text', 'language',
        'dialect_region', 'topic', 'sentiment', 'created_day'
    ]
    if c in frozen_test.columns
]

# إنشاء نسخة من بيانات الاختبار للاحتفاظ بها مع التوقعات
# Create a copy of the test data
test_predictions = frozen_test[keep].copy()

# حفظ التصنيفات الحقيقية لكل رسالة
# Store the actual topic labels
test_predictions['y_true'] = frozen_test['topic'].values

# تحويل الأرقام التي توقعها النموذج إلى أسماء التصنيفات
# Convert predicted numerical IDs into topic names
test_predictions['prediction'] = [labels[i] for i in test_ids]

# مقارنة التوقعات بالإجابات الحقيقية وحساب Macro-F1
# Compare predictions with true labels and calculate Macro-F1
test_f1 = f1_score(
    test_predictions.y_true,
    test_predictions.prediction,
    average='macro'
)


# ✏️ TODO 4: احسب الفرق بين نتيجة النموذج والأساس | difference = transformer score − baseline score
difference = test_f1 - baseline_f1

# عرض نتائج النموذجين والفرق بينهما
# Display the model scores and their difference
print(f'TF-IDF (validation)      : {baseline_f1:.3f}')
print(f'Transformer (frozen test): {test_f1:.3f}')
print(f'Difference / الفرق       : {difference:+.3f}   (الهدف / target ≥ +0.08)')

# حفظ توقعات النموذج على بيانات الاختبار في ملف CSV
# Save test predictions to a CSV file
test_predictions.to_csv(
    ARTIFACT_DIR / 'day2_topic_test_predictions.csv',
    index=False
)

# توقعات النموذج على بيانات validation
valid_output = trainer.predict(valid_ds)
valid_ids = np.argmax(valid_output.predictions, axis=1)

keep = [c for c in ['feedback_id', 'citizen_id', 'text', 'text_clean', 'language',
                    'dialect_region', 'topic', 'sentiment', 'created_day']
        if c in valid.columns]

valid_predictions = valid[keep].copy()
valid_predictions['y_true'] = valid['topic'].values
valid_predictions['prediction'] = [labels[i] for i in valid_ids]

# نحتفظ بالصفوف اللي أخطأ فيها النموذج فقط
validation_errors = valid_predictions[valid_predictions.y_true != valid_predictions.prediction]

validation_errors.to_csv(ARTIFACT_DIR / 'day2_validation_errors.csv', index=False)
print('Validation errors:', len(validation_errors), 'من', len(valid_predictions))

Map:   0%|          | 0/2394 [00:00<?, ? examples/s]

TF-IDF (validation)      : 0.820
Transformer (frozen test): 0.876
Difference / الفرق       : +0.056   (الهدف / target ≥ +0.08)


Validation errors: 168 من 1362


In [ ]:
import os, shutil
from google.colab import files

src = '/content/bayan_topic_model'
dst = '/content/bayan_topic_model_final'
os.makedirs(dst, exist_ok=True)

# ننسخ الملفات الموجودة مباشرة في المجلد فقط (config, model, tokenizer)
for f in os.listdir(src):
    p = os.path.join(src, f)
    if os.path.isfile(p):
        shutil.copy(p, dst)

shutil.make_archive(dst, 'zip', dst)
files.download(dst + '.zip')

KeyboardInterrupt: 

## أسئلة للنقاش | Think and discuss
1. لماذا التقسيم العشوائي للصفوف خطر؟ / Why is a random row split risky?
2. أي زوج مواضيع الأكثر خلطًا عندكم؟ / Which topic pair is most confused?
3. لماذا لا نلمس الاختبار المجمّد حتى النهاية؟ / Why keep the frozen test untouched?


## 📝 ملخص اللاب | Lab Summary

### ماذا فعلنا؟ | What did we do?

| # | الخطوة | Step | الفكرة الأساسية | Key idea |
|---|---|---|---|---|
| 1 | تحميل البيانات | Load data | استخدمنا البيانات المنظّفة من Lab 1 دون إعادة تنظيفها | Used the cleaned data from Lab 1, no re-cleaning |
| 2 | التقسيم حسب المواطن | Split by citizen | كل مواطن في قسم واحد فقط (تدريب / تحقق / اختبار مجمّد) لمنع التسريب | Each citizen in one split only, to prevent leakage |
| 3 | النموذج الأساسي TF-IDF | TF-IDF baseline | نموذج بسيط نقيس به: هل النموذج القوي أفضل فعلًا؟ | A simple yardstick: is the strong model really better? |
| 4 | تدريب Transformer | Train the Transformer | أكملنا تدريب نموذج مدرَّب مسبقًا (Transfer Learning) على بياناتنا | Fine-tuned a pretrained model on our data |
| 5 | الاختبار النهائي والحفظ | Final test and save | اختبرنا مرة واحدة على الاختبار المجمّد وحفظنا التوقعات والأخطاء | Tested once on the frozen test and saved predictions and errors |
